# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata attributes
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"Keywords: {dataset.metadata.keywords}")
print(f"Version: {dataset.metadata.version}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Retrieve all record sets and show their @id and label
print("Available Record Sets (by @id):")
for record_set in dataset.record_sets:
    print(f"- {record_set['@id']}: {record_set.get('name', '')}")

# For each record set, show its fields and columns
for record_set in dataset.record_sets:
    print(f"\nRecord Set: {record_set['@id']} ({record_set.get('name', '')})")
    fields = record_set.get('field', [])
    if not isinstance(fields, list):
        fields = [fields]
    for field in fields:
        # field can be a dict with @id or a string
        fid = field['@id'] if isinstance(field, dict) and '@id' in field else field
        # Try to find more information about this field
        field_obj = next((f for f in dataset.fields if f['@id'] == fid), None)
        if field_obj:
            label = field_obj.get('name', '')
            col_ref = field_obj.get('column', None)
            print(f"  - Field: {fid}, name: {label}, column: {col_ref}")
        else:
            print(f"  - Field: {fid}")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Choose the main data record set by id (based on the overview above, most often it is the primary RecordSet for experimental data)
# You may adjust record_set_id below based on the previous cell's output
main_record_set_id = None
for record_set in dataset.record_sets:
    if 'experiment' in record_set['@id'].lower() or 'main' in record_set['@id'].lower():
        main_record_set_id = record_set['@id']
        break
if not main_record_set_id:
    # fallback: take the first
    main_record_set_id = dataset.record_sets[0]['@id']

# For demonstration, collect all available record set ids
record_sets = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}
for record_set_id in record_sets:
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        print(f"Loaded {len(df)} records from RecordSet {record_set_id}")
        dataframes[record_set_id] = df
    except Exception as e:
        print(f"Could not load records from {record_set_id}: {e}")

# Display column names for the main record set
print(f"\nColumns in main record set ({main_record_set_id}):")
print(dataframes[main_record_set_id].columns.tolist())
# Show first 5 rows
dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section should include operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Select a numeric field for analysis.
# Adjust the field id (as a column name) according to data overview.
df = dataframes[main_record_set_id]

# Try to auto-detect a numeric field (e.g., 'molecular_weight' or similar)
# Fallback or manual selection if field names are unusual.
numeric_candidates = [col for col in df.columns if df[col].dtype in ['float', 'int'] or pd.api.types.is_numeric_dtype(df[col])]
if numeric_candidates:
    numeric_field = numeric_candidates[0]
else:
    # Guess from typical names
    for cand in df.columns:
        if 'weight' in cand.lower() or 'abundance' in cand.lower() or 'count' in cand.lower():
            numeric_field = cand
            break
    else:
        print('No obvious numeric field found. Please update numeric_field variable.')
        numeric_field = df.columns[0]

print(f"Using numeric field: {numeric_field}")

# Choose threshold as the mean (or median) for demonstration
threshold = df[numeric_field].mean() if pd.api.types.is_numeric_dtype(df[numeric_field]) else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
print(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} for filtered records:")
print(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try to select a group field for demo (e.g., 'accession'/'gene'/'sample'), else pick first object/categorical column
group_candidates = [col for col in df.columns if pd.api.types.is_object_dtype(df[col]) and col != numeric_field]
group_field = group_candidates[0] if group_candidates else df.columns[0]

if group_field in df.columns:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
    print(f"Grouped data by {group_field}:")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), kde=True, color='skyblue')
plt.xlabel(numeric_field)
plt.title(f"Distribution of {numeric_field}")
plt.show()

# Boxplot by the group_field
if group_field in df.columns and df[group_field].nunique() < 20:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=df[group_field], y=df[numeric_field])
    plt.title(f"{numeric_field} by {group_field}")
    plt.show()

# Correlation heatmap for numeric fields
if len(numeric_candidates) > 1:
    plt.figure(figsize=(8,6))
    sns.heatmap(df[numeric_candidates].corr(), annot=True, cmap='viridis')
    plt.title("Correlation Matrix of Numeric Fields")
    plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR<sup>2</sup> Croissant dataset for mass spectrometry analysis of extracellular vesicles was successfully loaded and explored using `mlcroissant`.
- Main record set and fields were identified using their `@id`s.
- Numeric field distributions and group-wise summaries revealed patterns in the dataset that can inform downstream analysis, such as biomarker discovery or sample comparison.
- Please refer to the dataset's Croissant metadata for in-depth provenance and licensing information.